<a href="https://colab.research.google.com/github/hamza-24-ai/Pytorch_with_practical_deepLearning/blob/main/10_LSTM_Next_word_predictor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install nltk

In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from nltk.tokenize import word_tokenize
import nltk
from collections import Counter

In [3]:
dataset = """
What is the course fee for  Data Science Mentorship Program (DSMP 2.0)
The total course fee for the DSMP course is Rs 10,600. This includes the fee for both DSMP 1.0 and DSMP 2.0

What is the total duration of the course?
The total duration of the course is about 8-10 months.

Are Deep Learning and NLP a part of the DSMP 2.0  course?
No, NLP and Deep Learning both are not a part of this program’s curriculum. Please follow our FREE YouTube playlist for these topics.

Is this course live? Will I get a recording of the session?
This course ended in Jan 2025 and all our sessions are recorded, you can login into our portal and watch the recording as per your convenience.

What is the time duration of all the live sessions?
Roughly, all the sessions last 2 hours.

What is the language spoken by the instructor during the sessions?
Hinglish.

How will I be informed about the upcoming class?
You will get mail from our side before every session once you enroll in the course.

Can I do this course if I am from a non-tech background?
Yes, absolutely. However, you’ll need to be consistent.



Where do I have to submit the task?
You don’t have to submit the task. We will provide you with the solutions, you have to self evaluate the task yourself.

Will we do case studies in the program?
Yes, there are 5 end to end case studies and multiple smaller projects.

Do we offer any student discounts or promo codes?
We do offer discounts on our courses a few times a year, and we announce them through our social media channels.

On how many devices can I watch the videos simultaneously?
You can watch the videos on one device at a time.

Do you provide notes for the lectures?
We do not provide notes for our lectures. We encourage students to make their own notes as it aids in study retention.

What is the difference between being a paid member on your YouTube channel and your website?
While the content on our YouTube channel and website is similar, enrolling through our website offers additional benefits such as doubt clearance support and access to an exclusive community of learners on Discord and more.

Do I get access to your YouTube channel as a paid member after enrolling through your website?
No, the YouTube channel membership is separate from what we offer on our website.

Is there any upcoming batch for the Data Science Mentorship Program?
Currently, we do not have any plans to launch a new batch for DSMP. We recommend you join DSMP2.

Do you provide doubt clearance for the projects uploaded on CampusX YouTube channel?
We do not provide doubt clearance for any of our videos or projects uploaded on YouTube. However, you can ask your doubts in our Discord channel or in the comments section.

Payment/Registration related questions

Where do we have to make our payments? Your YouTube channel or website?
You have to make all your monthly payments on our website. Here is the link for our website - https://learnwith.campusx.in/

What if I don’t like the course after making the payment? What is the refund policy?
You get a 7 days refund period from the day you have made the payment. To apply for refund, send us an email at support@campusx.in



I am living outside India and I am not able to make the payment on the website. What should I do?
You can contact us by sending a mail at support@campusx.in

I am from Pakistan and I am unable to make the payment?
We currently do not have the option to accept payment from Pakistan directly. If you’d still like to enroll for the course, you may ask a friend or family member living outside Pakistan to make the fee payment on your behalf.

Do you accept PayPal as a payment method?
Yes, we do. Please contact us at support@campusx.in for more details on using PayPal for payment.

I cannot pay the DSMP 2.0 fee as a one-time payment, is there any way to make a payment in installments?
At the moment, we do not have a monthly payment plan for the DSMP 2.0 course. However, if you have financial constraints, you can enroll in DSMP 1.0 by paying 799/- a month. Once you complete the course, you may apply for a discount coupon from the website.


Post registration queries

Till when can I view the paid videos on the website?
You can watch the videos till your subscription is valid, that is 3 years from the date of purchase for DSMP 2.0 and 2 years for DSMP 1.0

Why is lifetime validity not provided?
Because of the low course fee.

Do you assist with personal projects?
We do not provide support for personal projects or 1:1 mentorship due to time constraints and current commitments.


Where can I reach out in case of a doubt after a session?
You will have to fill a google form provided in your dashboard and our team will contact you for a 1 on 1 doubt clearing session.


What is the validity of Doubt Support?
You will get doubt support for 1 year from the data of registration.

Certificate and Placement Assistance related queries

What is the criteria to get the certificate?
There are 2 criterias:
You have to pay the entire fee of Rs 10,600
You have to attempt all the course assessments.

I have read that Job Preparation is a part of this program. What comes under Job Preparation?
This is to clarify that Job Preparation does not mean Placement guarantee. We don't guarantee our learners any jobs or for that matter even interview calls. If you are planning to join this course just for placements, we’re afraid that this course might not be the best for you.

Here is what comes under Job Preparation
Portfolio Building sessions.
Interview Questions.
Sessions with industry mentors.
Discussion on Job hunting strategies.

"""

In [4]:
# Tokenization
nltk.download("punkt")
nltk.download("punkt_tab")

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [5]:
# tokenize
tokens = word_tokenize(dataset.lower())

In [6]:
# build vocabulary
vocab = {"<UNK>" : 0}

for token in Counter(tokens).keys():
  if token not in vocab:
    vocab[token] = len(vocab)

print(f"Length of Vocab : {len(vocab)}")


Length of Vocab : 342


In [7]:
input_sentence = dataset.lower().split("\n")

In [8]:
# convert data into indices

def convert_into_indices(sentence, vocab):
  numerical_sentence = []
  for token in sentence:
    if token in vocab:
      numerical_sentence.append(vocab[token])
    else:
      numerical_sentence.append(vocab["<UNK>"])

  return numerical_sentence

In [9]:
input_numerical_sentence = []

for sentence in input_sentence:
  input_numerical_sentence.append(convert_into_indices(word_tokenize(sentence), vocab))


In [10]:
# making the sequence of data

sequence_learning = []
for sentence in input_numerical_sentence:

  for i in range(1,len(sentence)):
    sequence_learning.append(sentence[:i+1])

In [11]:
len(sequence_learning)

1054

In [12]:
# calculate the maximum len of squence learning
len_sequence = []

for seq in sequence_learning:
  len_sequence.append(len(seq))

max(len_sequence)

56

In [19]:
# padding the data to make it equal sentences by adding zeros
padded_sequence = []

for length in sequence_learning:
  padded_sequence.append([0]*(max(len_sequence)-len(length)) + length)


len(padded_sequence[2])

56

In [20]:
# convert into tensor

padded_sequence = torch.tensor(padded_sequence, dtype=torch.long)

In [21]:
padded_sequence

tensor([[  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        [  0,   0,   0,  ...,   2,   3,   4],
        ...,
        [  0,   0,   0,  ..., 136, 314, 340],
        [  0,   0,   0,  ..., 314, 340, 341],
        [  0,   0,   0,  ..., 340, 341,  29]])

In [23]:
# getting input and output
X = padded_sequence[:,:-1]
y = padded_sequence[:,-1]


In [24]:
X

tensor([[  0,   0,   0,  ...,   0,   0,   1],
        [  0,   0,   0,  ...,   0,   1,   2],
        [  0,   0,   0,  ...,   1,   2,   3],
        ...,
        [  0,   0,   0,  ..., 339, 136, 314],
        [  0,   0,   0,  ..., 136, 314, 340],
        [  0,   0,   0,  ..., 314, 340, 341]])

In [25]:
y

tensor([  2,   3,   4,  ..., 340, 341,  29])

In [27]:
# making custom dataset
class CustomDataset(Dataset):

  def __init__(self, X,y):
    self.X = X
    self.y = y

  def __len__(self):
    return self.X.shape[0]

  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]


In [28]:
dataset = CustomDataset(X,y)

In [29]:
len(dataset)

1054

In [30]:
# making dataloader

dataloader = DataLoader(dataset, batch_size=32, shuffle=True)

In [31]:
# Making Long Short Term memory

class LSTM(nn.Module):
  def __init__(self, vocab_size):

    super().__init__()

    self.embedded = nn.Embedding(vocab_size, 100)
    self.lstm = nn.LSTM(100, 150, batch_first=True)
    self.linear = nn.Linear(150, vocab_size)

  def forward(self,x):
    emb = self.embedded(x)
    intermediate_hidden, (final_hidden, self_state) = self.lstm(emb)
    output = self.linear(final_hidden.squeeze(0))

    return output


In [32]:
model = LSTM(len(vocab))

In [34]:
device = ("cuda" if torch.cuda.is_available() else "cpu")

In [35]:
model.to(device)

LSTM(
  (embedded): Embedding(342, 100)
  (lstm): LSTM(100, 150, batch_first=True)
  (linear): Linear(in_features=150, out_features=342, bias=True)
)

In [37]:
epochs = 50
learning_rate = 0.001

criterian = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

In [40]:
# make training model

for epoch in range(epochs):
  total_loss = 0

  for X,y in dataloader:
    X_batch = X.to(device)
    y_batch = y.to(device)

    optimizer.zero_grad()

    output = model(X_batch)

    loss = criterian(output, y_batch)

    loss.backward()

    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch : {epoch + 1} -> Loss : {total_loss}")




Epoch : 1 -> Loss : 187.92438650131226
Epoch : 2 -> Loss : 165.8246784210205
Epoch : 3 -> Loss : 152.89222812652588
Epoch : 4 -> Loss : 139.71128702163696
Epoch : 5 -> Loss : 126.61714816093445
Epoch : 6 -> Loss : 113.74861860275269
Epoch : 7 -> Loss : 101.63837337493896
Epoch : 8 -> Loss : 90.65306377410889
Epoch : 9 -> Loss : 80.27831101417542
Epoch : 10 -> Loss : 70.99139487743378
Epoch : 11 -> Loss : 62.420114398002625
Epoch : 12 -> Loss : 54.5144659280777
Epoch : 13 -> Loss : 47.54914164543152
Epoch : 14 -> Loss : 41.42636829614639
Epoch : 15 -> Loss : 36.14213454723358
Epoch : 16 -> Loss : 31.567266166210175
Epoch : 17 -> Loss : 27.76629400253296
Epoch : 18 -> Loss : 24.3398717045784
Epoch : 19 -> Loss : 21.623065382242203
Epoch : 20 -> Loss : 19.329873740673065
Epoch : 21 -> Loss : 17.303667694330215
Epoch : 22 -> Loss : 15.689597219228745
Epoch : 23 -> Loss : 14.23928028345108
Epoch : 24 -> Loss : 12.973347634077072
Epoch : 25 -> Loss : 11.964345246553421
Epoch : 26 -> Loss : 1

In [45]:

# make prdiction model

def prediction(model, vocab, text):
  # make tokenize
  tokenized = word_tokenize(text)

  # make indices
  numerical = convert_into_indices(tokenized, vocab)

  # padding
  pad = torch.tensor([0]*(56 - len(numerical)) + numerical, dtype=torch.long).unsqueeze(0).to(device)

  # predict from model

  output = model(pad)

  # make into 1 dimension
  value, index = torch.max(output, dim=1)

  return text + " " + list(vocab.keys())[index]


In [49]:
prediction(model, vocab, "The course follows a monthly")

'The course follows a monthly is'

In [50]:
# predict a sequence of word
import time

input = "I am from"
data = 10

for i in range(data):
  output = prediction(model, vocab, input)
  print(output)
  input = output
  time.sleep(0.5)

I am from pakistan
I am from pakistan and
I am from pakistan and i
I am from pakistan and i am
I am from pakistan and i am unable
I am from pakistan and i am unable to
I am from pakistan and i am unable to make
I am from pakistan and i am unable to make the
I am from pakistan and i am unable to make the payment
I am from pakistan and i am unable to make the payment ?


In [51]:
datalaoder1 = DataLoader(dataset, batch_size=32, shuffle=False)

In [54]:
# evaluation the model

def calculate_accuracy(model, dataloader, device):
  model.eval()
  correct = 0
  total = 0

  with torch.no_grad():

    for batch_x, batch_y in dataloader:
      batch_x, batch_y = batch_x.to(device), batch_y.to(device)

      output = model(batch_x)

      _,predicted = torch.max(output, dim=1)

      correct += (predicted == batch_y).sum().item()
      total += batch_y.size(0)

  accuracy = correct/total * 100
  return accuracy

print(f"Accuracy of this model : {calculate_accuracy(model, datalaoder1, device):.2f}%")



Accuracy of this model : 96.11%
